## Alphavantage Glue Job (tariff_alphavantage_glue_job)
#### Silver → Gold Iceberg Pipeline

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 1. Args, Globals & Imports

In [16]:
import sys, json
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from awsglue.utils import getResolvedOptions

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
args = getResolvedOptions(
    sys.argv,
    ["domain", "source", "dataset", "keys", "run_id", "dag_id"]
)

DOMAIN = ARGS["domain"]
SOURCE = ARGS["source"]
DATASET = ARGS["dataset"]
KEYS = json.loads(ARGS["keys"]) # list of relative paths/keys to ingested data files
RECORD_COUNT = int(ARGS["record_count"])
INGESTED_AT = ARGS["ingested_at"]
DAG_ID = ARGS["dag_id"]
RUN_ID = ARGS["run_id"]

print(f"Running glue job for DAG: {DAG_ID}, run_id: {RUN_ID}, ingested_at: {INGESTED_AT}")

if not KEYS:
    raise ValueError("No bronze files provided to Silver job")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
GLUE_CATALOG = "glue_catalog"
WAREHOUSE = spark.conf.get("spark.sql.catalog.glue_catalog.warehouse")
SILVER_DB = "silver"
GOLD_DB = "gold"

SILVER_TABLE = f"{DOMAIN}_{SOURCE}_{DATASET}_silver"
GOLD_TABLE = f"{DOMAIN}_{SOURCE}_{DATASET}_gold"

SILVER_FQN = f"{GLUE_CATALOG}.{SILVER_DB}.{SILVER_TABLE}"
GOLD_FQN = f"{GLUE_CATALOG}.{GOLD_DB}.{GOLD_TABLE}"

BRONZE_PATHS = [f"{WAREHOUSE}/{k}" for k in KEYS]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 2. Read Bronze

In [31]:
bronze_df = spark.read.option("mode", "FAILFAST").json(BRONZE_PATHS)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+-----------------+------+--------------------------------+------+-------+--------------------------------+-------------------------------------------------------------------------------------------------------------------------+------------+------+----------+--------+
|close |dataset          |high  |ingested_at                     |low   |open   |received_at                     |request_url                                                                                                              |source      |symbol|trade_date|volume  |
+------+-----------------+------+--------------------------------+------+-------+--------------------------------+-------------------------------------------------------------------------------------------------------------------------+------------+------+----------+--------+
|260.33|time_series_daily|263.68|2026-01-08T06:01:33.891759+00:00|259.81|263.2  |2026-01-08T06:01:31.462683+00:00|https://www.alphavantage.co/query?function=TIME_SERIES_

In [ ]:
print("Running Bronze validations...")

if bronze_df.count() == 0:
    raise RuntimeError("Bronze validation failed: no records found")

required_cols = {
    "symbol",
    "trade_date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "ingested_at"
}

missing = required_cols - set(bronze_df.columns)
if missing:
    raise RuntimeError(f"Bronze validation failed: missing columns {missing}")

print("Bronze validations passed")

## 3. Transform to Silver

In [32]:
silver_incoming = (
    bronze_df
    # ----------------------------
    # Business keys
    # ----------------------------
    .withColumn("symbol", col("symbol"))
    .withColumn("trade_date", to_date(col("trade_date")))

    # ----------------------------
    # OHLCV metrics
    # ----------------------------
    .withColumn("open", col("open").cast("double"))
    .withColumn("high", col("high").cast("double"))
    .withColumn("low", col("low").cast("double"))
    .withColumn("close", col("close").cast("double"))
    .withColumn("volume", col("volume").cast("bigint"))

    # ----------------------------
    # Source metadata
    # ----------------------------
    .withColumn("source", col("source"))
    .withColumn("dataset", col("dataset"))
    .withColumn("request_url", col("request_url"))

    # ----------------------------
    # Timestamps
    # ----------------------------
    .withColumn("received_at", to_timestamp(col("received_at")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at")))

    # ----------------------------
    # SCD2 record hash
    # ----------------------------
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("symbol"),
                col("trade_date").cast("string"),
                col("open"),
                col("high"),
                col("low"),
                col("close"),
                col("volume")
            ),
            256
        )
    )

    # ----------------------------
    # SCD2 control columns
    # ----------------------------
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))

    # ----------------------------
    # Lineage & orchestration
    # ----------------------------
    .withColumn("dag_id", lit(DAG_ID))
    .withColumn("run_id", lit(RUN_ID))
    .withColumn("processed_at", current_timestamp())
)

silver_incoming.createOrReplaceTempView("incoming")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 4. Deduplicate incoming batch

In [34]:
w = Window.partitionBy("symbol", "trade_date").orderBy(col("ingested_at").desc())
silver_dedup = (
  silver_incoming
  .withColumn("rn", row_number().over(w))
  .filter(col("rn") == 1)
  .drop("rn")
)

silver_dedup.createOrReplaceTempView("incoming_dedup")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
print("Running incoming batch validations...")

dup_cnt = spark.sql("""
SELECT COUNT(*) AS cnt
FROM (
  SELECT symbol, trade_date
  FROM incoming_dedup
  GROUP BY symbol, trade_date
  HAVING COUNT(*) > 1
)
""").collect()[0]["cnt"]

if dup_cnt > 0:
    raise RuntimeError(f"Incoming validation failed: {dup_cnt} duplicate business keys after deduplication")

print("Incoming validations passed")

## 5. Merge Silver

In [36]:
spark.sql(f"""
MERGE INTO {SILVER_FQN} t
USING incoming_dedup s
ON t.symbol = s.symbol AND t.trade_date = s.trade_date AND t.is_current = true
WHEN MATCHED AND t.record_hash <> s.record_hash THEN
  UPDATE SET t.effective_to = s.effective_from, t.is_current = false
WHEN NOT MATCHED THEN INSERT *
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]

In [42]:
print("Running Silver SCD2 invariant checks...")

multiple_currents = spark.sql(f"""
SELECT symbol, trade_date
FROM {SILVER_FQN}
WHERE is_current = true
GROUP BY symbol, trade_date
HAVING COUNT(*) > 1
""").count()

if multiple_currents > 0:
    raise ValueError(f"Found current records ({multiple_currents} records) for single business keys (symbol, trade_date) indicating FAULTY MERGE")

missing_currents = spark.sql(f"""
SELECT symbol, trade_date
FROM {SILVER_FQN}
GROUP BY symbol, trade_date
HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) = 0
""").count()

if missing_currents > 0:
    raise ValueError(f"Found records that are missing ({missing_currents} records) for single business keys (symbol, trade_date) indicating DATA LOSS and FAULTY MERGE")
    
print("Silver validations passed")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
'>' not supported between instances of 'function' and 'int'
Traceback (most recent call last):
TypeError: '>' not supported between instances of 'function' and 'int'



## 6. Rebuild Gold

In [39]:
spark.sql(f"""
INSERT OVERWRITE {GOLD_FQN}
SELECT
    symbol,
    trade_date,

    open,
    high,
    low,
    close,
    volume,

    current_timestamp() AS processed_at
FROM {SILVER_FQN}
WHERE is_current = true
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[]